In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score


In [5]:

strategies = [
    "fivecrop",
    "tencrop",
    "random5",
    "multiscale"
]

base_dir = "saved_models_multicrop"

seeds = [0,1,2,3,4]

results = []

def recall_at_k(sim_matrix, query_labels, gallery_labels, k=1):
    correct = 0

    for i in range(len(query_labels)):
        sims = sim_matrix[i]
        idx = np.argsort(-sims)[:k]

        if query_labels[i] in gallery_labels[idx]:
            correct += 1

    return correct / len(query_labels)



In [6]:

for strategy in strategies:

    data = np.load(f"{base_dir}/{strategy}/data.npz")

    X = data["embeddings"]
    y = data["label_ids"]
    print("Strategy:", strategy)
    print("Embeddings shape:", X.shape)
    print("Labels shape:", y.shape)
    for seed in seeds:

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.3,
            stratify=y,
            random_state=seed
        )

        sim = cosine_similarity(X_test, X_train)

        # nearest neighbor prediction
        nn_idx = np.argmax(sim, axis=1)
        y_pred = y_train[nn_idx]

        acc = accuracy_score(y_test, y_pred)

        r1 = recall_at_k(sim, y_test, y_train, k=1)
        r5 = recall_at_k(sim, y_test, y_train, k=5)

        results.append({
            "strategy": strategy,
            "seed": seed,
            "accuracy": acc,
            "recall@1": r1,
            "recall@5": r5
        })



Strategy: fivecrop
Embeddings shape: (319, 768)
Labels shape: (319,)
Strategy: tencrop
Embeddings shape: (319, 768)
Labels shape: (319,)
Strategy: random5
Embeddings shape: (319, 768)
Labels shape: (319,)
Strategy: multiscale
Embeddings shape: (319, 768)
Labels shape: (319,)


In [7]:

df = pd.DataFrame(results)

summary = df.groupby("strategy")[["accuracy","recall@1","recall@5"]].mean()

print("\nResults per split:")
display(df.sort_values("accuracy", ascending=False))

print("\nAverage performance:")
display(summary.sort_values("accuracy", ascending=False))



Results per split:


,strategy,seed,accuracy,recall@1,recall@5
4,fivecrop,4,0.718750,0.718750,0.864583
9,tencrop,4,0.718750,0.718750,0.864583
1,fivecrop,1,0.666667,0.666667,0.843750
3,fivecrop,3,0.666667,0.666667,0.802083
6,tencrop,1,0.666667,0.666667,0.843750
8,tencrop,3,0.666667,0.666667,0.802083
0,fivecrop,0,0.614583,0.614583,0.854167
5,tencrop,0,0.614583,0.614583,0.854167
11,random5,1,0.614583,0.614583,0.812500
14,random5,4,0.604167,0.604167,0.843750



Average performance:


,accuracy,recall@1,recall@5
strategy,,,
fivecrop,0.652083,0.652083,0.831250
tencrop,0.652083,0.652083,0.831250
random5,0.581250,0.581250,0.802083
multiscale,0.566667,0.566667,0.777083
